# Render Parameter Shifting

Orchestrator: dispatches `templates/parameter_shifting.ipynb` via papermill
for each free parameter in the BaseCMR model bounds.

In [ ]:
import papermill as pm
from cru_to_cmr.helpers import find_project_root
from cru_to_cmr.config import base_model_configs

ROOT = find_project_root()

In [ ]:
bounds = base_model_configs["BaseCMR"]

shared_params = {
    "model_name": "BaseCMR",
    "model_factory_path": "cru_to_cmr.models.cmr_compare.CMRFactory",
    "fit_result_name": "HealeyKahana2014_BaseCMR_Fitting.json",
    "data_name": "HealeyKahana2014",
    "data_query": "data['listtype'] == -1",
    "experiment_count": 50,
    "run_tag": "Parameter_Shifting",
    "seed": 0,
    "analysis_paths": [
        "cru_to_cmr.analyses.spc.plot_spc",
        "cru_to_cmr.analyses.crp.plot_crp",
        "cru_to_cmr.analyses.pnr.plot_pnr",
    ],
}

# Optional: set to a parameter name to run only that sweep, or None for all
target_parameter = None

In [ ]:
template = f"{ROOT}/analyses/templates/parameter_shifting.ipynb"
rendered_dir = f"{ROOT}/analyses/rendered"

for param_name, (lo, hi) in bounds.items():
    if target_parameter is not None and param_name != target_parameter:
        continue

    output_path = f"{rendered_dir}/shifting_{param_name}.ipynb"

    params = shared_params | {
        "varied_parameter": param_name,
        "sweep_min": lo,
        "sweep_max": hi,
    }

    print(f"Rendering: {param_name}")
    pm.execute_notebook(
        template,
        output_path,
        parameters=params,
        progress_bar=True,
    )
    print(f"  -> {output_path}")